In [1]:
import sys
!{sys.executable} -m pip install --upgrade scikit-learn imbalanced-learn

import sys
!{sys.executable} -m pip install xgboost

import os
import random
from pathlib import Path
from IPython.display import display, HTML

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import sklearn, imblearn
print("sklearn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
from scipy.stats import pointbiserialr
from imblearn.over_sampling import RandomOverSampler
import sys
!{sys.executable} -m pip install --upgrade scikit-learn

import time
import datetime
from datetime import datetime
from timeit import default_timer as timer
from dateutil.relativedelta import relativedelta

import joblib
from joblib import Parallel, delayed, parallel_backend

from sklearn.utils import resample
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif, chi2, RFE
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, f1_score

# — assume X_imputed (DataFrame), y (Series), and SEED are already defined —



from imblearn.over_sampling import SMOTE
from xgboost import XGBRegressor, XGBClassifier

plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline
%config InlineBackend.figure_format='retina'

# set random seed for reproducibility
def set_seeds(seed=777):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

SEED = 777
set_seeds(SEED)
print(f'Random seed set to: {SEED}')

# checking
set_seeds(SEED)
print(np.random.rand(3))
set_seeds(SEED)
print(np.random.rand(3))

sklearn: 1.6.1
imbalanced-learn: 0.13.0
Random seed set to: 777
[0.15266373 0.30235661 0.06203641]
[0.15266373 0.30235661 0.06203641]


# Load Data

In [2]:
data_path = '/Users/jihoonchoi/Dropbox/PhD/Python_personal/df_dataset.csv'
merged_df = pd.read_csv(data_path)
merged_df['respondent_id'] = merged_df['respondent_id'].astype(str).str.replace('.0', '', regex=False)
merged_df.set_index('respondent_id', inplace=True)
merged_df

,is_sleep_deprived,daily_sleep_duration,sleep_risk_category,gender,age_years,race_hispanic_origin,race_hispanic_origin_with_asian,exam_period,country_of_birth,education_level,...,minutes_sedentary_activity,sedentary_hours_per_day,sedentary_level,smoking_status,smokers_in_house_category,smokers_inside_category,secondhand_smoke_exposure,tobacco_use_type,bmi_category,weight_change_category
respondent_id,,,,,,,,,,,,,,,,,,,,,
109266,0,7.66,Low Risk,Female,29.0,Other/Multi-Racial,Non-Hispanic Asian,May-Oct,Foreign Born,College Graduate+,...,480.0,8.0,moderate,never,none,none,moderate,none,obese_class2,moderate_gain
109267,0,8.00,Low Risk,Female,21.0,Other Hispanic,Other Hispanic,NaN,Foreign Born,Some College/AA,...,540.0,9.0,high,never,none,none,moderate,NaN,normal,stable
109268,0,8.37,Low Risk,Female,18.0,Non-Hispanic White,Non-Hispanic White,NaN,US Born,NaN,...,540.0,9.0,high,never,none,none,moderate,NaN,normal,stable
109271,0,10.87,Low Risk,Male,49.0,Non-Hispanic White,Non-Hispanic White,May-Oct,US Born,9-11th Grade,...,60.0,1.0,low,current_everyday,two,two,high,cigs_only,obese_class1,large_gain
109273,1,6.99,High Risk,Male,36.0,Non-Hispanic White,Non-Hispanic White,May-Oct,US Born,Some College/AA,...,180.0,3.0,low,current_everyday,one,none,moderate,cigs_only,normal,moderate_gain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142305,0,9.00,Low Risk,Female,76.0,Mexican American,Mexican American,May-Oct,Foreign Born,Less than 9th,...,480.0,8.0,moderate,never,none,none,moderate,none,overweight,stable
142307,0,7.00,Low Risk,Female,49.0,Non-Hispanic Black,Non-Hispanic Black,May-Oct,US Born,College Graduate+,...,480.0,8.0,moderate,former,two,none,moderate,none,obese_class2,large_loss
142308,0,8.73,Low Risk,Male,50.0,Other Hispanic,Other Hispanic,Nov-Apr,Foreign Born,Some College/AA,...,600.0,10.0,high,never,none,none,moderate,none,overweight,moderate_loss


In [3]:
# X, y
to_exlucde = ['daily_sleep_duration', 'wake_time_weekdays_usual_hours', 'wake_time_weekends_usual_hours']
X = merged_df.drop(columns=['is_sleep_deprived'] + to_exlucde)
y = merged_df['is_sleep_deprived']

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (18438, 193)
y shape: (18438,)


## Filter Method + Wrapper Method
- No imputation yet
- Filter method + Wrapper method -> Union
- If over 30, further selection based on feature importance (XGBoost)

# Feature Selection

In [4]:
# remove columns with more than 80% missing values
threshold = 0.8
missing_ratio = X.isnull().mean()
keep_cols = missing_ratio[missing_ratio < threshold].index
X_filtered = X[keep_cols].copy()
print(f'Removed {len(X.columns) - len(X_filtered.columns)} columns with >80% missing values')
print('Shape after missing value filtering:', X_filtered.shape)

Removed 0 columns with >80% missing values
Shape after missing value filtering: (18438, 193)


In [5]:
# 1. filter method
# split data into numeric and categorical columns
numeric_cols = X_filtered.select_dtypes(include=[np.number]).columns
categorical_cols = X_filtered.select_dtypes(exclude=[np.number]).columns

# filter numeric variables
# fill missing values with 0 temporarily
X_numeric = X_filtered[numeric_cols].fillna(0)
# select top 30 numeric features
selector_numeric = SelectKBest(score_func=f_classif, k=30) # f_classif
X_numeric_filtered = selector_numeric.fit_transform(X_numeric, y)
# get names of selected numeric features
numeric_selected_filter = numeric_cols[selector_numeric.get_support()].tolist()
print('\nSelected numeric features from Filter (30):', numeric_selected_filter)

# filter categorical variables
# fill missing values with the most frequent value
X_categorical = X_filtered[categorical_cols].fillna(X_filtered[categorical_cols].mode().iloc[0])
# convert categorical values to numeric codes
X_categorical_encoded = X_categorical.astype('category').apply(lambda x: x.cat.codes)
# select top 10 categorical features
selector_categorical = SelectKBest(score_func=chi2, k=10) # chi2
X_categorical_filtered = selector_categorical.fit_transform(X_categorical_encoded, y)
# get names of selected categorical features
categorical_selected_filter = categorical_cols[selector_categorical.get_support()].tolist()
print('Selected categorical features from Filter (10):', categorical_selected_filter)

# total selected features
filter_selected = numeric_selected_filter + categorical_selected_filter # combine filter results (numeric + categorical)
print('\nTotal features from Filter method (40):', filter_selected)
print('Shape after Filter method:', X_filtered[filter_selected].shape)

# 2. wrapper method
X_encoded = X_filtered.copy()
# convert categorical columns to numeric codes
for col in categorical_cols:
    X_encoded[col] = X_encoded[col].astype('category').cat.codes

# rfe with xgboost model
model = XGBClassifier(
    eval_metric='logloss',
    random_state=SEED,
    enable_categorical=False # use numeric codes, not categories
)
# rfe to select top 40 features
rfe = RFE(estimator=model, n_features_to_select=40, step=5)
rfe.fit(X_encoded, y)
# selected features
wrapper_selected = X_encoded.columns[rfe.support_].tolist()
print('\nSelected features from Wrapper (RFE, 40):', wrapper_selected)
print('Shape after Wrapper method:', X_filtered[wrapper_selected].shape)

# 3. combine features
# combine filter and wrapper features (union, remove duplicates)
combined_features = list(set(filter_selected + wrapper_selected))
print('\nCombined features (Filter U Wrapper):', combined_features)
# intersection
#combined_features = list(set(filter_selected) & set(wrapper_selected))
#print('\nCombined features (Filter ∩ Wrapper):', combined_features)
print('Shape after combining:', X_filtered[combined_features].shape)

# 4. reduce to top 30 features
if len(combined_features) > 30:
    X_combined_encoded = X_filtered[combined_features].copy()
    # convert categorical columns to numeric codes
    for col in categorical_cols:
        if col in X_combined_encoded.columns:
            X_combined_encoded[col] = X_combined_encoded[col].astype('category').cat.codes
    # xgboost to get feature importance
    model.fit(X_combined_encoded, y)
    # get top 30 features based on importance
    importances = pd.Series(model.feature_importances_, index=X_combined_encoded.columns)
    top_30_features = importances.sort_values(ascending=False).head(30).index.tolist()
    # select final data with top 30 features
    X_selected = X_filtered[top_30_features].copy()
    print('\nFinal 30 features after additional selection:')
    print(top_30_features)
    print('Final X shape:', X_selected.shape)
else:
    # use combined features as final data
    X_selected = X_filtered[combined_features].copy()
    print('\nFinal features (no additional selection needed):', combined_features)
    print('Final X shape:', X_selected.shape)


Selected numeric features from Filter (30): ['weight_kg', 'body_mass_index', 'sleep_time_weekdays_usual_hours', 'sleep_time_weekends_usual_hours', 'depression_binary', 'sleep_problems_flag', 'fatigue_flag', 'sleep_disturbance_score', 'poor_oral_health', 'current_smoker', 'num_smokers_inside_home', 'num_smokers_in_house', 'any_smokers_in_house', 'any_indoor_smoking', 'used_cigs_last_5_days', 'used_any_tobacco_last_5_days', 'smoked_tobacco_last_5_days', 'weight_last_1_year_pounds', 'current_weight_pounds', 'bmi', 'feeling_down_last_2_weeks', 'trouble_sleeping_last_2_weeks', 'poor_appetite_or_overeating_last_2_weeks', 'moving_speaking_slow_or_fast_last_2_weeks', 'little_interest_last_2_weeks', 'feeling_bad_about_self_last_2_weeks', 'feeling_tired_last_2_weeks', 'phq9_total', 'job_difficulty_due_to_mouth_last_year', 'teeth_gums_health_rating']
Selected categorical features from Filter (10): ['sleep_risk_category', 'gender', 'marital_status', 'depression_severity', 'smoking_status', 'smoke

In [6]:
# data types and missing values for selected features
print('\nFinal X dtypes and missing values:')
print(pd.concat([X_selected.dtypes, X_selected.isnull().sum()], axis=1))


Final X dtypes and missing values:
                                             0     1
sleep_time_weekdays_usual_hours        float64     0
gender                                  object     0
race_hispanic_origin                    object     0
age_years                              float64     0
work_type_last_week                     object     5
sleep_risk_category                     object     0
ever_told_copd                         float64     0
sleep_time_weekends_usual_hours        float64     0
race_hispanic_origin_with_asian         object     0
trouble_sleeping_last_2_weeks          float64  4790
smoking_status                          object   868
teeth_gums_health_rating               float64    19
weight_change_category                  object     0
feeling_down_last_2_weeks              float64  4793
education_level                         object  1651
estrone_pmol_L                         float64  5125
sleep_time_consistency                 float64     0
sleep_dist

In [7]:
# X_selected = X_filtered[top_30_features].copy() # reset

## Test Prediction
- preliminary evaluation

In [8]:
# train test split
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=SEED) # 8:2

# encode categorical variables to numeric codes (for xgboost)
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()
# loop through object-type columns and convert to numeric codes
for col in X_selected.select_dtypes(include=['object']).columns:
    X_train_encoded[col] = X_train_encoded[col].astype('category').cat.codes
    X_test_encoded[col] = X_test_encoded[col].astype('category').cat.codes

# xgboost classifier
model = XGBClassifier(eval_metric='logloss', random_state=SEED)
model.fit(X_train_encoded, y_train) # train on encoded training data
# predict on encoded test data
y_pred = model.predict(X_test_encoded)
print(classification_report(y_test, y_pred)) # classification performance metrics

              precision    recall  f1-score   support

           0       0.83      0.92      0.87      2911
           1       0.47      0.27      0.34       777

    accuracy                           0.78      3688
   macro avg       0.65      0.59      0.61      3688
weighted avg       0.75      0.78      0.76      3688



# Handle Missing Values
- Imputation using XGBRegressor and XGBClassifier

In [9]:
# select numerical and categorical cols
numerical_cols = X_selected.select_dtypes(include=['float64']).columns # numerical
categorical_cols = X_selected.select_dtypes(include=['object']).columns # categorical

# encode categorical cols (for imputation)
x_categorical_encoded = X_selected[categorical_cols].copy()
for col in categorical_cols:
    x_categorical_encoded[col] = x_categorical_encoded[col].astype('category').cat.codes
    x_categorical_encoded[col] = x_categorical_encoded[col].replace(-1, np.nan) # replace -1 (missing values after encoding) with nan

In [10]:
# impute numerical cols (iterativeimputer)
numerical_imputer = IterativeImputer(
    estimator=XGBRegressor(random_state=SEED), # iterativeimputer using xgboost
    max_iter=20, # iterations 10 -> 20 to help convergence
    tol=1e-2, # relax tolerance for convergence
    random_state=SEED
)
x_numerical_imputed = numerical_imputer.fit_transform(X_selected[numerical_cols])
x_numerical_imputed_df = pd.DataFrame(
    x_numerical_imputed,
    columns=numerical_cols,
    index=X_selected.index
)

# impute categorical columns (iterativeimputer)
categorical_imputer = IterativeImputer(
    estimator=XGBClassifier(random_state=SEED), # iterativeimputer using xgboost
    max_iter=20,
    tol=1e-2,
    random_state=SEED
)
x_categorical_imputed = categorical_imputer.fit_transform(x_categorical_encoded)
x_categorical_imputed_df = pd.DataFrame(
    x_categorical_imputed,
    columns=categorical_cols,
    index=X_selected.index
)

# decode categorical columns back to original values
x_categorical_final = x_categorical_imputed_df.copy()
for col in categorical_cols:
    # map numeric codes back to original categories
    code_to_category = dict(enumerate(X_selected[col].astype('category').cat.categories))
    x_categorical_final[col] = x_categorical_final[col].round().astype(int).map(code_to_category)

# combine numerical and categorical imputed data
X_imputed = pd.concat([x_numerical_imputed_df, x_categorical_final], axis=1)
X_imputed.head()


/Users/jihoonchoi/anaconda3/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(
/Users/jihoonchoi/anaconda3/lib/python3.11/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


,sleep_time_weekdays_usual_hours,age_years,ever_told_copd,sleep_time_weekends_usual_hours,trouble_sleeping_last_2_weeks,teeth_gums_health_rating,feeling_down_last_2_weeks,estrone_pmol_L,sleep_time_consistency,sleep_disturbance_score,...,race_hispanic_origin,work_type_last_week,sleep_risk_category,race_hispanic_origin_with_asian,smoking_status,weight_change_category,education_level,marital_status,smokers_inside_category,smokers_in_house_category
respondent_id,,,,,,,,,,,,,,,,,,,,,
109266,22.0,29.0,0.0,23.0,5.397605e-79,4.0,5.397605e-79,168.000000,1.0,0.0,...,Other/Multi-Racial,working,Low Risk,Non-Hispanic Asian,never,moderate_gain,College Graduate+,Never Married,none,none
109267,0.0,21.0,0.0,3.0,9.890120e-06,1.0,5.064917e-01,8674.681641,3.0,0.0,...,Other Hispanic,working,Low Risk,Other Hispanic,never,stable,Some College/AA,Never Married,none,none
109268,22.0,18.0,0.0,23.0,9.890120e-06,2.0,7.846866e-01,381.898926,1.0,0.0,...,Non-Hispanic White,not_working,Low Risk,Non-Hispanic White,never,stable,College Graduate+,Married/Partner,none,none
109271,23.0,49.0,1.0,23.0,5.397605e-79,4.0,1.000000e+00,551.000000,0.0,0.0,...,Non-Hispanic White,not_working,Low Risk,Non-Hispanic White,current_everyday,large_gain,9-11th Grade,Never Married,two,two
109273,8.0,36.0,0.0,21.0,2.000000e+00,5.0,2.000000e+00,95.500000,13.0,4.0,...,Non-Hispanic White,working,High Risk,Non-Hispanic White,current_everyday,moderate_gain,Some College/AA,Never Married,none,one


In [11]:
# data types and missing values for selected features
X_imputed.to_csv('X_imputed_dk.csv', index=False)

print('Imputed data dtypes and missing values:')
print(pd.concat([X_imputed.dtypes, X_imputed.isnull().sum()], axis=1))

Imputed data dtypes and missing values:
                                             0  1
sleep_time_weekdays_usual_hours        float64  0
age_years                              float64  0
ever_told_copd                         float64  0
sleep_time_weekends_usual_hours        float64  0
trouble_sleeping_last_2_weeks          float64  0
teeth_gums_health_rating               float64  0
feeling_down_last_2_weeks              float64  0
estrone_pmol_L                         float64  0
sleep_time_consistency                 float64  0
sleep_disturbance_score                float64  0
age_first_period                       float64  0
job_difficulty_due_to_mouth_last_year  float64  0
standing_height_cm                     float64  0
weight_last_1_year_pounds              float64  0
feeling_tired_last_2_weeks             float64  0
minutes_sedentary_activity             float64  0
family_income_to_poverty_ratio         float64  0
estradiol_pmol_L                       float64  0
poor_oral_

# Modeling and Test Prediction
- XGBClassifier

In [12]:
# encode categorical cols to numeric codes (before splitting)
X_encoded = X_imputed.copy()

X_encoded.to_csv('X_encoded.csv', index=False)

categorical_cols = X_imputed.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X_encoded[col] = X_encoded[col].astype('category').cat.codes # convert to category and get codes
    X_encoded[col] = X_encoded[col].replace(-1, np.nan) # handle any -1 values (missing after encoding)

# calculate scale_pos_weight for class imbalance
class_counts = y.value_counts()
# set weight for minority class to balance classes (class 0 / class 1)
scale_pos_weight = class_counts[0] / class_counts[1] # ratio of class 0 to class 1

# train/test split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=SEED, stratify=y
)

# xgboost with tuned parameters
model = XGBClassifier(
    eval_metric='logloss',
    random_state=SEED,
    enable_categorical=False, # use numeric codes instead of categories
    scale_pos_weight=scale_pos_weight, # increase weight for minority class
    max_depth=5, # limit tree depth to avoid overfitting
    learning_rate=0.1, # slower learning for better generalization
    n_estimators=200 # use more trees for better fit
)
model.fit(X_train, y_train)

# predict on test set
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred)) # classification performance

              precision    recall  f1-score   support

           0       0.88      0.77      0.82      2885
           1       0.43      0.63      0.51       803

    accuracy                           0.74      3688
   macro avg       0.65      0.70      0.66      3688
weighted avg       0.78      0.74      0.75      3688



# Cross Validation
- cv f1-score for class 1 

In [13]:
# 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = []
for train_idx, val_idx in skf.split(X_encoded, y):
    # split data with stratified folds to keep class ratio
    X_cv_train, X_cv_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_cv_train, y_cv_val = y.iloc[train_idx], y.iloc[val_idx]
    # train model with same parameters
    cv_model = XGBClassifier(
        eval_metric='logloss',
        random_state=SEED,
        enable_categorical=False,
        scale_pos_weight=scale_pos_weight,
        max_depth=5,
        learning_rate=0.1,
        n_estimators=200
    )
    cv_model.fit(X_cv_train, y_cv_train)
    # predict on validation set
    y_cv_pred = cv_model.predict(X_cv_val)
    # calculate f1-score for each fold
    cv_scores.append(f1_score(y_cv_val, y_cv_pred))

# cross-validation results
print('5-fold stratified cv f1-score:')
print(f'mean: {np.mean(cv_scores):.3f}, std: {np.std(cv_scores):.3f}')

# feature importance
feature_importance = pd.Series(model.feature_importances_, index=X_encoded.columns)
print('\ntop 10 feature importance:')
print(feature_importance.sort_values(ascending=False).head(10))

5-fold stratified cv f1-score:
mean: 0.509, std: 0.009

top 10 feature importance:
sleep_time_weekdays_usual_hours    0.199869
gender                             0.052924
age_years                          0.044746
work_type_last_week                0.037948
race_hispanic_origin               0.036514
sleep_disturbance_score            0.031290
sleep_time_weekends_usual_hours    0.030734
teeth_gums_health_rating           0.030311
marital_status                     0.029241
trouble_sleeping_last_2_weeks      0.028874
dtype: float32


In [15]:
# 1) Encode any object‐dtype columns as codes
X_nn = X_imputed.copy()
for col in X_nn.select_dtypes(include=['object']):
    X_nn[col] = X_nn[col].astype('category').cat.codes.replace(-1, np.nan)

# 2) Compute sample weights to up‐weight the minority class
counts = y.value_counts()
weight = counts[0] / counts[1]
sample_weight = np.where(y == counts.index[1], weight, 1)

# 1. Split & scale as before
X_train, X_test, y_train, y_test = train_test_split(
    X_nn, y, test_size=0.2, stratify=y, random_state=SEED
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# 2. Oversample minority in the training set
ros = RandomOverSampler(random_state=SEED)
X_res, y_res = ros.fit_resample(X_train, y_train)

# 3. Fit the neural net on the balanced data
nn = MLPClassifier(hidden_layer_sizes=(100,), max_iter=200, random_state=SEED)
nn.fit(X_res, y_res)

# 4. Evaluate on the untouched test set
y_pred = nn.predict(X_test)
print(classification_report(y_test, y_pred))



              precision    recall  f1-score   support

           0       0.84      0.78      0.81      2885
           1       0.37      0.47      0.42       803

    accuracy                           0.71      3688
   macro avg       0.61      0.63      0.61      3688
weighted avg       0.74      0.71      0.72      3688



/Users/jihoonchoi/anaconda3/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


TypeError: BaseMultilayerPerceptron.fit() got an unexpected keyword argument 'sample_weight'

In [16]:
# 7) 5‐fold stratified CV for F1
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = []
for train_idx, val_idx in skf.split(X_nn, y):
    X_tr, X_val = scaler.fit_transform(X_nn.iloc[train_idx]), scaler.transform(X_nn.iloc[val_idx])
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    sw_tr        = sample_weight[train_idx]
    
    m = MLPClassifier(
        hidden_layer_sizes=(100,),
        activation='relu',
        solver='adam',
        max_iter=200,
        random_state=SEED
    )
    m.fit(X_tr, y_tr, sample_weight=sw_tr)
    cv_scores.append(f1_score(y_val, m.predict(X_val)))

print(f"5-fold CV F1-score mean: {np.mean(cv_scores):.3f}, std: {np.std(cv_scores):.3f}")

TypeError: BaseMultilayerPerceptron.fit() got an unexpected keyword argument 'sample_weight'